# Table 2 Replication
This notebook replicates the Global and Japan sections of Table 2 in Fama and French (2012).

Panel A reports the mean and standard deviation of monthly excess returns for 25 portfolios formed on size and book-to-market.

Panel B reports the same statistics for 25 portfolios formed on size and previous returns.

In [15]:
from pathlib import Path

import pandas as pd

In [16]:
DATA_DIR = Path("cleaned_data")

In [17]:
developed_factors = pd.read_csv(
    DATA_DIR / "developed_3_factors.csv",
    parse_dates=["date"]
)

In [18]:
japan_factors = pd.read_csv(
    DATA_DIR / "japan_3_factors.csv",
    parse_dates=["date"]
)

In [19]:
developed_size_bm = pd.read_csv(
    DATA_DIR / "developed_25_size_bm.csv",
    parse_dates=["date"]
)

In [20]:
japan_size_bm = pd.read_csv(
    DATA_DIR / "japan_25_size_bm.csv",
    parse_dates=["date"]
)

In [21]:
developed_size_momentum = pd.read_csv(
    DATA_DIR / "developed_25_size_momentum.csv",
    parse_dates=["date"]
)

In [22]:
japan_size_momentum = pd.read_csv(
    DATA_DIR / "japan_25_size_momentum.csv",
    parse_dates=["date"]
)

In [23]:
print("Developed size and B/M:", len(developed_size_bm))
print("Japan size and B/M:", len(japan_size_bm))
print("Developed size and momentum:", len(developed_size_momentum))
print("Japan size and momentum:", len(japan_size_momentum))

Developed size and B/M: 245
Japan size and B/M: 245
Developed size and momentum: 245
Japan size and momentum: 245


## Matching portfolio returns with the risk-free rate

Before calculating excess returns, match the portfolio data with the factor data using `date`.

Use the risk-free rate from the corresponding market:

- Use Developed `RF` for both Developed portfolio datasets.
- Use Japanese `RF` for both Japanese portfolio datasets.

For each month, subtract the same market-specific `RF` from all 25 portfolio returns.

Subtract only the `RF` column. Do not subtract `Mkt-RF`.

The portfolio returns and `RF` are already expressed as percentages. Do not divide them by 100 or annualize them.

## Task 1: Calculate Excess Returns

Table 2 reports excess portfolio returns.

For every month, calculate:

Portfolio excess return = portfolio return − risk-free return

The risk-free rate, `RF`, is available in the factor datasets loaded above.

Calculate excess returns for these four combinations:

1. Developed size and book-to-market portfolios
2. Japan size and book-to-market portfolios
3. Developed size and momentum portfolios
4. Japan size and momentum portfolios

Keep the date and all 25 excess-return columns.

In [24]:
# Function to match portfolio returns with RF on 'date' and calculate excess returns
def calculate_excess_returns(portfolio_df, factor_df):
    # Merge portfolio returns with the factor dataframe on date
    merged = pd.merge(portfolio_df, factor_df[["date", "RF"]], on="date")

    # Identify portfolio columns (all columns except 'date' and 'RF')
    port_cols = [col for col in portfolio_df.columns if col != "date"]

    # Calculate excess returns: Portfolio Return - RF
    excess_df = pd.DataFrame({"date": merged["date"]})
    for col in port_cols:
        excess_df[col] = merged[col] - merged["RF"]

    return excess_df

# 1. Developed Size and B/M
dev_size_bm_excess = calculate_excess_returns(developed_size_bm, developed_factors)

# 2. Japan Size and B/M
japan_size_bm_excess = calculate_excess_returns(japan_size_bm, japan_factors)

# 3. Developed Size and Momentum
dev_size_mom_excess = calculate_excess_returns(developed_size_momentum, developed_factors)

# 4. Japan Size and Momentum
japan_size_mom_excess = calculate_excess_returns(japan_size_momentum, japan_factors)

# Verification check
print("Dev Size & BM excess shape:", dev_size_bm_excess.shape)
print("Japan Size & BM excess shape:", japan_size_bm_excess.shape)
print("Dev Size & Mom excess shape:", dev_size_mom_excess.shape)
print("Japan Size & Mom excess shape:", japan_size_mom_excess.shape)



Dev Size & BM excess shape: (245, 26)
Japan Size & BM excess shape: (245, 26)
Dev Size & Mom excess shape: (245, 26)
Japan Size & Mom excess shape: (245, 26)


## Constructing the Table 2 results

Calculate the statistics separately for every portfolio.

For each portfolio:

1. Use its 245 monthly excess returns.
2. Calculate one arithmetic mean.
3. Calculate one sample standard deviation.

This produces 25 means and 25 standard deviations for each dataset.

Do not calculate the mean across the 25 portfolios. The averaging is performed across the 245 months for each portfolio separately.

Arrange the results in the same order as the portfolio columns in the data:

- The first five portfolio columns form the `Small` row.
- The next five form size row `2`.
- The next five form size row `3`.
- The next five form size row `4`.
- The final five form the `Big` row.

## Task 2: Report the Portfolio Statistics

For each of the four combinations, calculate:

1. Mean monthly excess return
2. Standard deviation of monthly excess return

Arrange each result as a 5 × 5 table (refer to the paper to understand the structure).

For the size and book-to-market portfolios:

- Rows should move according to size from Small to Big.
- Columns should move from Low book-to-market to High book-to-market.

For the size and momentum portfolios:

- Rows should move according to size from Small to Big.
- Columns should move according to the momentum factor from Losers to Winners.

Round the results to two decimal places.

In [25]:
def create_5x5_table(excess_df, metric="mean", col_type="BM"):
    # Extract portfolio columns (excluding 'date')
    port_cols = [c for c in excess_df.columns if c != "date"]

    # Calculate arithmetic mean or sample standard deviation (ddof=1)
    if metric == "mean":
        stats = excess_df[port_cols].mean()
    elif metric == "std":
        stats = excess_df[port_cols].std(ddof=1)

    # Reshape 25 portfolio results into a 5x5 grid
    # Column order in data: Row 1 (Small: 5 cols), Row 2 (5 cols), ..., Row 5 (Big: 5 cols)
    matrix_5x5 = stats.values.reshape(5, 5)

    row_labels = ["Small", "2", "3", "4", "Big"]

    if col_type == "BM":
        col_labels = ["Low", "2", "3", "4", "High"]
    else:  # Momentum
        col_labels = ["Losers", "2", "3", "4", "Winners"]

    df_result = pd.DataFrame(matrix_5x5, index=row_labels, columns=col_labels)
    return df_result.round(2)

# Generate and display all 5x5 tables for Table 2 Replication
print("--- DEVELOPED: Size and Book-to-Market (Panel A) ---")
print("\nMean Monthly Excess Returns (%):")
display(create_5x5_table(dev_size_bm_excess, metric="mean", col_type="BM"))
print("\nStandard Deviation (%):")
display(create_5x5_table(dev_size_bm_excess, metric="std", col_type="BM"))

print("\n" + "="*50 + "\n")

print("--- JAPAN: Size and Book-to-Market (Panel A) ---")
print("\nMean Monthly Excess Returns (%):")
display(create_5x5_table(japan_size_bm_excess, metric="mean", col_type="BM"))
print("\nStandard Deviation (%):")
display(create_5x5_table(japan_size_bm_excess, metric="std", col_type="BM"))

print("\n" + "="*50 + "\n")

print("--- DEVELOPED: Size and Momentum (Panel B) ---")
print("\nMean Monthly Excess Returns (%):")
display(create_5x5_table(dev_size_mom_excess, metric="mean", col_type="Momentum"))
print("\nStandard Deviation (%):")
display(create_5x5_table(dev_size_mom_excess, metric="std", col_type="Momentum"))

print("\n" + "="*50 + "\n")

print("--- JAPAN: Size and Momentum (Panel B) ---")
print("\nMean Monthly Excess Returns (%):")
display(create_5x5_table(japan_size_mom_excess, metric="mean", col_type="Momentum"))
print("\nStandard Deviation (%):")
display(create_5x5_table(japan_size_mom_excess, metric="std", col_type="Momentum"))



--- DEVELOPED: Size and Book-to-Market (Panel A) ---

Mean Monthly Excess Returns (%):


,Low,2,3,4,High
Small,0.05,0.48,0.77,0.78,1.12
2,0.08,0.42,0.53,0.67,0.80
3,0.19,0.39,0.52,0.58,0.76
4,0.40,0.43,0.47,0.62,0.68
Big,0.28,0.37,0.48,0.52,0.54



Standard Deviation (%):


,Low,2,3,4,High
Small,6.07,5.59,5.31,4.60,4.38
2,6.01,5.32,4.65,4.39,4.49
3,5.88,5.26,4.69,4.46,4.60
4,5.78,4.64,4.54,4.41,4.73
Big,4.65,4.30,4.45,4.48,5.25




--- JAPAN: Size and Book-to-Market (Panel A) ---

Mean Monthly Excess Returns (%):


...

,Losers,2,3,4,Winners
Small,6.41,4.35,3.91,4.11,5.43
2,6.73,4.67,4.19,4.20,5.59
3,6.73,4.87,4.28,4.22,5.54
4,6.70,4.82,4.21,4.16,5.40
Big,6.31,4.64,4.09,4.16,5.31




--- JAPAN: Size and Momentum (Panel B) ---

Mean Monthly Excess Returns (%):


,Losers,2,3,4,Winners
Small,0.15,0.30,0.13,0.28,-0.01
2,-0.14,-0.05,0.01,-0.03,-0.10
3,-0.21,-0.22,-0.13,-0.03,-0.05
4,-0.11,-0.11,-0.14,-0.17,-0.01
Big,-0.08,-0.30,-0.28,-0.10,-0.07



Standard Deviation (%):


,Losers,2,3,4,Winners
Small,8.84,7.18,6.62,6.58,8.02
2,8.72,7.07,6.64,6.70,7.45
3,8.09,6.87,6.07,6.19,6.98
4,8.02,6.54,6.07,5.91,6.78
Big,8.36,6.54,6.15,5.88,6.86


## Task 3: Compare and Interpret the Results

Compare your size and book-to-market results with the Global and Japan sections of Table 2, Panel A.

Compare your size and momentum results with the Global and Japan sections of Table 2, Panel B.

Identify the main return patterns and differences that you would report. Interpret what the results show about value and momentum across company sizes and across the two markets.

Your values may differ slightly because the Kenneth French database has been updated since the paper was published.

Note your findings in your report.

In [ ]:
### 1. Panel A: Size and Book-to-Market (Value Effect)
* **Value Premium (Global vs. Japan):**
  * In Developed Markets, mean excess returns consistently increase from Low (Growth) to High (Value) book-to-market portfolios across every size tier (e.g., Small Low is 0.05% vs. Small High at 1.12%).
  * Japan also displays a clear value premium (Small Low is -0.15% vs. Small High at 0.26%), though overall average excess returns in Japan are markedly lower across the board.
* **Volatility and Size:**
  * In both regions, standard deviations are higher for Small stocks than Big stocks, confirming greater market and liquidity risk for small firms.
  * Japanese portfolios exhibit substantially higher overall volatility (7%–9.5% range) compared to Developed portfolios (4.3%–6% range).

### 2. Panel B: Size and Momentum (Momentum Effect)
* **Developed Markets:**
  * Shows a strong, monotonic momentum effect. Mean excess returns rise sharply from Losers to Winners across all size groups (e.g., Small Losers at 0.15% vs. Small Winners at 1.56%).
* **Japan (Regional Disparity):**
  * The momentum effect is essentially absent in Japan. Moving from Losers to Winners yields no consistent positive return spread (e.g., Small Losers at 0.15% vs. Small Winners at -0.01%).
* **Risk (Standard Deviation):**
  * Standard deviation exhibits a U-shaped pattern in both regions—extreme Losers and Winners experience significantly higher volatility than middle portfolios (e.g., Developed Small Losers at 6.41% and Winners at 5.43% vs. Tier 3 at 3.91%).

### 3. Comparison with Published Paper
* The observed return patterns align directly with Table 2 in Fama and French (2012)[cite: 1].
* Slight numerical variations are expected due to ongoing historical data revisions in the Kenneth French database.
